In [1]:
# Cell 1 — Imports & Environment Check
import subprocess, sys

def pip_install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["openpyxl", "transformers", "accelerate"]:
    try:
        __import__(pkg.replace("-","_"))
    except ImportError:
        print(f"Installing {pkg}...")
        pip_install(pkg)

# ── Standard library ──────────────────────────────────────────────
import os, json, re, ast, warnings, logging
import xml.etree.ElementTree as ET
from pathlib import Path
from collections import defaultdict, Counter

# ── Scientific stack ──────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── sklearn metrics ───────────────────────────────────────────────
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (f1_score,
                             label_ranking_average_precision_score,
                             label_ranking_loss,
                             coverage_error)

# ── PyTorch ───────────────────────────────────────────────────────
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast

# ── HuggingFace ───────────────────────────────────────────────────
from transformers import AutoTokenizer, AutoModel

# ── Misc ──────────────────────────────────────────────────────────
import openpyxl
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)

# ── Environment ───────────────────────────────────────────────────
print(f"Python  : {sys.version.split()[0]}")
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}  "
      f"({'  ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'})")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device  : {DEVICE}")

import random
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Config ────────────────────────────────────────────────────────
from types import SimpleNamespace
CFG = SimpleNamespace(
    encoder_name      = "basel/ATTACK-BERT",
    max_len           = 512,
    batch_size        = 64,       # inference only, can be large
    num_workers       = 0,
    pin_memory        = False,

    # Paths
    attack_json       = "OSRs/ATTACK/enterprise-attack-v16.1.json",
    kev_json          = "OSRs/kev-07.28.2025_attack-16.1-enterprise.json",
    smet_xlsx         = "CVE_annotated_dataset.xlsx",
    smet_id2mitre_url = "https://raw.githubusercontent.com/basel-a/SMET/main/id2mitre.json",

    # Eval
    top_k_list        = [1, 3, 5, 10],   # R@K values to report
)

# Validate paths
print("\nPaths:")
for label, path in [("ATT&CK STIX", CFG.attack_json),
                    ("KEV gold",    CFG.kev_json),
                    ("SMET xlsx",   CFG.smet_xlsx)]:
    ok = Path(path).exists()
    size = f"{Path(path).stat().st_size/1e6:.1f} MB" if ok else "MISSING"
    print(f"  {'✓' if ok else '✗'}  {label:<14} {path}  ({size})")

print("\n✓ Ready for Cell 2.")

c:\Users\OA\Desktop\New Work\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python  : 3.11.0
PyTorch : 2.11.0+cu126
CUDA    : True  (  NVIDIA GeForce RTX 3090)
Device  : cuda

Paths:
  ✓  ATT&CK STIX    OSRs/ATTACK/enterprise-attack-v16.1.json  (40.8 MB)
  ✓  KEV gold       OSRs/kev-07.28.2025_attack-16.1-enterprise.json  (1.2 MB)
  ✓  SMET xlsx      CVE_annotated_dataset.xlsx  (0.1 MB)

✓ Ready for Cell 2.


In [2]:
# Cell 2 — ATT&CK Label Space + Technique Text Builder

with open(CFG.attack_json, encoding="utf-8") as f:
    stix_bundle = json.load(f)

objects    = stix_bundle.get("objects", [])
stix_by_id = {o["id"]: o for o in objects}

# ── 1. Helper: get T-code from STIX object ────────────────────────
def get_tcode(obj):
    for ref in obj.get("external_references", []):
        if ref.get("source_name") == "mitre-attack":
            return ref.get("external_id", "")
    return ""

# ── 2. All attack-patterns ────────────────────────────────────────
all_techniques  = {}   # tcode → stix obj
technique_names = {}   # tcode → name
technique_descs = {}   # tcode → STIX description text

for o in objects:
    if o.get("type") != "attack-pattern":
        continue
    tc = get_tcode(o)
    if not tc.startswith("T"):
        continue
    all_techniques[tc]  = o
    technique_names[tc] = o.get("name", "")
    technique_descs[tc] = o.get("description", "")

print(f"Total attack-patterns (incl. subs): {len(all_techniques)}")

# ── 3. parent_map, revoked_map ────────────────────────────────────
parent_map  = {}   # sub → parent tcode
revoked_map = {}   # old → new tcode

for o in objects:
    if o.get("type") != "relationship":
        continue
    rt = o.get("relationship_type", "")
    src = stix_by_id.get(o.get("source_ref", ""))
    tgt = stix_by_id.get(o.get("target_ref", ""))
    if not src or not tgt:
        continue
    tc_src = get_tcode(src)
    tc_tgt = get_tcode(tgt)
    if rt == "subtechnique-of" and tc_src and tc_tgt:
        parent_map[tc_src] = tc_tgt
    elif rt == "revoked-by" and tc_src and tc_tgt:
        revoked_map[tc_src] = tc_tgt

print(f"Sub→parent mappings : {len(parent_map)}")
print(f"Revoked mappings    : {len(revoked_map)}")

def resolve_to_parent(tc):
    tc = revoked_map.get(tc, tc)
    tc = parent_map.get(tc, tc)
    tc = revoked_map.get(tc, tc)
    return tc

# ── 4. Active parent techniques (no dot, not revoked) ─────────────
revoked_tcodes = set(revoked_map.keys())
for tc, obj in all_techniques.items():
    if obj.get("x_mitre_revoked", False) or obj.get("revoked", False):
        revoked_tcodes.add(tc)

parent_techniques = {
    tc: obj for tc, obj in all_techniques.items()
    if "." not in tc and tc not in revoked_tcodes
}
print(f"Active parent techniques: {len(parent_techniques)}")

# ── 5. Sub-technique names grouped by parent ──────────────────────
sub_names_by_parent = defaultdict(list)   # parent tcode → [sub names]
for tc, obj in all_techniques.items():
    if "." not in tc:
        continue
    parent_tc = parent_map.get(tc)
    if parent_tc and parent_tc in parent_techniques:
        sub_names_by_parent[parent_tc].append(obj.get("name", ""))

print(f"Parents with sub-techniques: "
      f"{sum(1 for v in sub_names_by_parent.values() if v)}")

# ── 6. Build technique text for encoding ─────────────────────────
# Format: "[T1190] Exploit Public-Facing Application.
#          <STIX description>.
#          Sub-techniques: SQLi, PHP injection, ..."
def build_technique_text(tc):
    name  = technique_names.get(tc, "")
    desc  = technique_descs.get(tc, "")
    # Clean STIX markdown citations like (Citation: ...)
    desc  = re.sub(r'\(Citation:[^)]+\)', '', desc).strip()
    # Truncate description to first 3 sentences to stay within token budget
    sents = re.split(r'(?<=[.!?])\s+', desc)
    desc_short = " ".join(sents[:3]).strip()
    subs  = sub_names_by_parent.get(tc, [])
    text  = f"[{tc}] {name}."
    if desc_short:
        text += f" {desc_short}"
    if subs:
        text += f" Sub-techniques: {', '.join(subs)}."
    return text

# Build index: sorted list of all 214 parent T-codes
TECHNIQUE_LIST  = sorted(parent_techniques.keys())
TECHNIQUE_INDEX = {tc: i for i, tc in enumerate(TECHNIQUE_LIST)}
NUM_TECHNIQUES  = len(TECHNIQUE_LIST)

technique_texts = [build_technique_text(tc) for tc in TECHNIQUE_LIST]

print(f"\nTechnique index size: {NUM_TECHNIQUES}")
print(f"\nSample technique texts:")
for tc in ["T1190", "T1059", "T1566", "T1078"]:
    if tc in TECHNIQUE_INDEX:
        txt = technique_texts[TECHNIQUE_INDEX[tc]]
        print(f"\n  [{tc}] {txt[:200]}{'...' if len(txt)>200 else ''}")

# Token length distribution
print(f"\nTechnique text lengths (chars):")
lens = [len(t) for t in technique_texts]
print(f"  min={min(lens)}  max={max(lens)}  "
      f"mean={np.mean(lens):.0f}  median={np.median(lens):.0f}")

print("\n✓ Label space built. Ready for Cell 3.")

Total attack-patterns (incl. subs): 799
Sub→parent mappings : 456
Revoked mappings    : 139
Active parent techniques: 214
Parents with sub-techniques: 96

Technique index size: 214

Sample technique texts:

  [T1190] [T1190] Exploit Public-Facing Application. Adversaries may attempt to exploit a weakness in an Internet-facing host or system to initially access a network. The weakness in the system can be a softwar...

  [T1059] [T1059] Command and Scripting Interpreter. Adversaries may abuse command and script interpreters to execute commands, scripts, or binaries. These interfaces and languages provide ways of interacting w...

  [T1566] [T1566] Phishing. Adversaries may send phishing messages to gain access to victim systems. All forms of phishing are electronically delivered social engineering. Phishing can be targeted, known as spe...

  [T1078] [T1078] Valid Accounts. Adversaries may obtain and abuse credentials of existing accounts as a means of gaining Initial Access, Persistenc

In [3]:
# Cell 3 — Load Encoder & Build Technique Index

# ── Fix double-prefix in technique texts ─────────────────────────
def build_technique_text(tc):
    name = technique_names.get(tc, "")
    desc = technique_descs.get(tc, "")
    desc = re.sub(r'\(Citation:[^)]+\)', '', desc).strip()
    sents = re.split(r'(?<=[.!?])\s+', desc)
    desc_short = " ".join(sents[:3]).strip()
    subs = sub_names_by_parent.get(tc, [])
    text = f"[{tc}] {name}."
    if desc_short:
        text += f" {desc_short}"
    if subs:
        text += f" Sub-techniques: {', '.join(subs)}."
    return text

# Rebuild cleanly
technique_texts = [build_technique_text(tc) for tc in TECHNIQUE_LIST]

# Verify fix
sample = technique_texts[TECHNIQUE_INDEX["T1190"]]
print(f"Fixed sample: {sample[:120]}")

# ── 1. Load tokenizer + encoder ───────────────────────────────────
print("\nLoading SecureBERT_Plus...", flush=True)
tokenizer = AutoTokenizer.from_pretrained(CFG.encoder_name)
encoder   = AutoModel.from_pretrained(CFG.encoder_name).to(DEVICE)
encoder.eval()

total_params = sum(p.numel() for p in encoder.parameters())
print(f"✓ Encoder loaded  params={total_params/1e6:.1f}M  "
      f"vocab={tokenizer.vocab_size:,}")

# ── 2. Encode helper ──────────────────────────────────────────────
@torch.no_grad()
def encode_texts(texts, batch_size=32, desc="Encoding"):
    """
    Tokenize and encode a list of texts.
    Returns L2-normalised CLS embeddings: (N, 768) float32 numpy array.
    """
    all_embs = []
    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start: start + batch_size]
        enc = tokenizer(
            batch_texts,
            max_length=CFG.max_len,
            padding=True,
            truncation=True,
            return_tensors="pt",
        )
        input_ids      = enc["input_ids"].to(DEVICE)
        attention_mask = enc["attention_mask"].to(DEVICE)
        with autocast():
            out = encoder(input_ids=input_ids,
                          attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]   # (B, 768)
        cls = F.normalize(cls.float(), p=2, dim=-1)
        all_embs.append(cls.cpu().numpy())
        if (start // batch_size + 1) % 10 == 0 or \
                start + batch_size >= len(texts):
            print(f"\r  {desc}: {min(start+batch_size, len(texts))}"
                  f"/{len(texts)}", end="", flush=True)
    print()
    return np.vstack(all_embs)   # (N, 768)

# ── 3. Encode all 214 technique texts ────────────────────────────
print("\nBuilding technique index...")
technique_embs = encode_texts(
    technique_texts,
    batch_size=32,
    desc="Techniques",
)   # (214, 768)  L2-normalised

print(f"Technique index shape : {technique_embs.shape}")
print(f"Norm check (should≈1) : "
      f"min={np.linalg.norm(technique_embs, axis=1).min():.4f}  "
      f"max={np.linalg.norm(technique_embs, axis=1).max():.4f}")

# ── 4. Similarity retrieval function ─────────────────────────────
def retrieve_top_k(query_embs, k=10):
    """
    query_embs : (N, 768) L2-normalised
    Returns    : scores (N, 214), indices (N, 214) sorted desc
    """
    # cosine sim = dot product of L2-normalised vectors
    sim = query_embs @ technique_embs.T          # (N, 214)
    ranked_idx   = np.argsort(-sim, axis=1)      # (N, 214) desc
    ranked_scores = np.take_along_axis(sim, ranked_idx, axis=1)
    return ranked_scores, ranked_idx

# ── 5. Build full similarity matrix helper for metrics ────────────
def get_score_matrix(query_embs):
    """Returns (N, NUM_TECHNIQUES) cosine similarity matrix."""
    return query_embs @ technique_embs.T         # (N, 214)

# ── 6. Recall@K helper ────────────────────────────────────────────
def recall_at_k(labels, scores, k):
    """
    labels : (N, C) binary
    scores : (N, C) similarity scores
    Returns macro-averaged R@K.
    """
    top_k = np.argsort(-scores, axis=1)[:, :k]
    hits, total = 0, 0
    for i in range(len(labels)):
        pos = set(np.where(labels[i])[0])
        if not pos:
            continue
        hits  += len(pos & set(top_k[i]))
        total += len(pos)
    return hits / total if total > 0 else 0.0

# ── 7. Quick sanity check — encode 3 test CVEs ───────────────────
test_cves = [
    "SQL injection vulnerability allows remote attacker to execute "
    "arbitrary SQL commands via the user input field.",
    "Buffer overflow in the FTP server allows remote code execution "
    "via a long USER command.",
    "Phishing campaign uses spoofed emails to steal credentials from "
    "corporate users.",
]
print("\nSanity check — top-3 techniques per test CVE:")
test_embs = encode_texts(test_cves, batch_size=8, desc="Test CVEs")
_, top_idx = retrieve_top_k(test_embs, k=3)
for i, cve_text in enumerate(test_cves):
    print(f"\n  CVE: {cve_text[:70]}...")
    for rank, idx in enumerate(top_idx[i]):
        tc   = TECHNIQUE_LIST[idx]
        name = technique_names[tc]
        print(f"    #{rank+1}  {tc}  {name}")

print("\n✓ Technique index ready. Ready for Cell 4.")

Fixed sample: [T1190] Exploit Public-Facing Application. Adversaries may attempt to exploit a weakness in an Internet-facing host or s

Loading SecureBERT_Plus...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 31094.38it/s]


✓ Encoder loaded  params=109.5M  vocab=30,527

Building technique index...
  Techniques: 214/214
Technique index shape : (214, 768)
Norm check (should≈1) : min=1.0000  max=1.0000

Sanity check — top-3 techniques per test CVE:
  Test CVEs: 3/3

  CVE: SQL injection vulnerability allows remote attacker to execute arbitrar...
    #1  T1659  Content Injection
    #2  T1190  Exploit Public-Facing Application
    #3  T1059  Command and Scripting Interpreter
    #4  T1212  Exploitation for Credential Access
    #5  T1202  Indirect Command Execution
    #6  T1203  Exploitation for Client Execution
    #7  T1064  Scripting
    #8  T1189  Drive-by Compromise
    #9  T1221  Template Injection
    #10  T1211  Exploitation for Defense Evasion
    #11  T1565  Data Manipulation
    #12  T1584  Compromise Infrastructure
    #13  T1055  Process Injection
    #14  T1204  User Execution
    #15  T1068  Exploitation for Privilege Escalation
    #16  T1586  Compromise Accounts
    #17  T1554  Compromise Ho

In [4]:
# Cell 4 — KEV + SMET Evaluation

# ── Shared metrics helpers ────────────────────────────────────────
def recall_at_k(labels, scores, k):
    top_k = np.argsort(-scores, axis=1)[:, :k]
    hits, total = 0, 0
    for i in range(len(labels)):
        pos = set(np.where(labels[i])[0])
        if not pos:
            continue
        hits  += len(pos & set(top_k[i]))
        total += len(pos)
    return hits / total if total > 0 else 0.0

def full_metrics(scores, labels, tag, k_list=(1,3,5,10)):
    lrap = label_ranking_average_precision_score(labels, scores)
    rl   = label_ranking_loss(labels, scores)
    ce   = coverage_error(labels, scores)
    print(f"\n── {tag} ──────────────────────────────────────────")
    print(f"  LRAP           : {lrap:.4f}")
    print(f"  Ranking Loss   : {rl:.4f}")
    print(f"  Coverage Error : {ce:.4f}")
    for k in k_list:
        rk = recall_at_k(labels, scores, k)
        print(f"  R@{k:<3}          : {rk:.4f}")
    return dict(lrap=lrap, ranking_loss=rl, coverage_error=ce,
                **{f"r{k}": recall_at_k(labels, scores, k)
                   for k in k_list})

# ══════════════════════════════════════════════════════════════════
# A. KEV EVALUATION
# ══════════════════════════════════════════════════════════════════

# ── 1. Load & parse KEV ───────────────────────────────────────────
with open(CFG.kev_json, encoding="utf-8") as f:
    kev_data = json.load(f)
kev_entries = kev_data["mapping_objects"]

# Group by CVE → set of parent T-codes
kev_by_cve = defaultdict(set)
kev_desc   = {}
for entry in kev_entries:
    cve_id = entry.get("capability_id", "").strip()
    tc_raw = entry.get("attack_object_id", "").strip()
    desc   = entry.get("capability_description", "").strip()
    if not cve_id.startswith("CVE-") or not tc_raw:
        continue
    parent_tc = resolve_to_parent(tc_raw)
    if parent_tc in TECHNIQUE_INDEX:
        kev_by_cve[cve_id].add(parent_tc)
    if desc and cve_id not in kev_desc:
        kev_desc[cve_id] = desc

# Build eval dataframe
kev_rows = [{"cve_id": cid, "description": kev_desc.get(cid, ""),
             "techniques": list(techs)}
            for cid, techs in kev_by_cve.items() if techs]
df_kev = pd.DataFrame(kev_rows)
print(f"KEV eval rows       : {len(df_kev)}")
print(f"Unique T-codes      : "
      f"{len(set(t for ts in df_kev['techniques'] for t in ts))}")

# Drop rows with empty description
df_kev = df_kev[df_kev["description"].str.len() > 10].reset_index(drop=True)
print(f"Rows with desc      : {len(df_kev)}")

# ── 2. Encode KEV descriptions ────────────────────────────────────
print("\nEncoding KEV descriptions...", flush=True)
kev_embs   = encode_texts(df_kev["description"].tolist(),
                           batch_size=CFG.batch_size, desc="KEV")
kev_scores = get_score_matrix(kev_embs)          # (N, 214)

# ── 3. Build label matrix ─────────────────────────────────────────
mlb_full = MultiLabelBinarizer(classes=TECHNIQUE_LIST)
mlb_full.fit([TECHNIQUE_LIST])
kev_labels = mlb_full.transform(
    df_kev["techniques"].tolist()).astype(np.float32)   # (N, 214)

print(f"kev_scores shape    : {kev_scores.shape}")
print(f"kev_labels shape    : {kev_labels.shape}")
print(f"label density       : {kev_labels.mean():.4f}")

# ── 4. KEV metrics ────────────────────────────────────────────────
r_kev = full_metrics(kev_scores, kev_labels,
                     "KEV zero-shot (n={})".format(len(df_kev)))

# ══════════════════════════════════════════════════════════════════
# B. SMET EVALUATION
# ══════════════════════════════════════════════════════════════════

# ── 5. Load SMET ──────────────────────────────────────────────────
df_smet = pd.read_excel(CFG.smet_xlsx, engine="openpyxl")

# ── 6. Build name→T-code map ──────────────────────────────────────
name_to_tcode = {}
for tc, obj in parent_techniques.items():
    name_to_tcode[obj.get("name","").strip().lower()] = tc
for tc, name in technique_names.items():
    parent_tc = resolve_to_parent(tc)
    if parent_tc in TECHNIQUE_INDEX:
        name_to_tcode[name.strip().lower()] = parent_tc

# id2mitre fallback
try:
    import urllib.request
    with urllib.request.urlopen(CFG.smet_id2mitre_url, timeout=10) as r:
        id2mitre = json.loads(r.read().decode("utf-8"))
    for k, v in id2mitre.items():
        kl = k.strip().lower()
        if isinstance(v, str) and re.match(r'T\d{4}', v):
            parent_tc = resolve_to_parent(v)
            if parent_tc in TECHNIQUE_INDEX:
                name_to_tcode[kl] = parent_tc
    print(f"\nid2mitre loaded: {len(id2mitre)} entries")
except Exception as e:
    print(f"\nid2mitre fetch failed: {e}")

print(f"name→T-code entries : {len(name_to_tcode)}")

def parse_smet_techniques(val):
    if pd.isna(val):
        return []
    try:
        names = ast.literal_eval(str(val))
    except Exception:
        names = [str(val)]
    return [name_to_tcode[n.strip().lower()]
            for n in names if n.strip().lower() in name_to_tcode]

df_smet["tcodes"] = df_smet["ATT&CK Techniques"].apply(parse_smet_techniques)
df_smet_eval = df_smet[df_smet["tcodes"].apply(len) > 0].reset_index(drop=True)

print(f"SMET rows mapped    : {len(df_smet_eval)} / {len(df_smet)}")
unique_smet_tc = set(t for ts in df_smet_eval["tcodes"] for t in ts)
print(f"Unique T-codes      : {len(unique_smet_tc)}  {sorted(unique_smet_tc)}")

# ── 7. Encode SMET descriptions ───────────────────────────────────
print("\nEncoding SMET descriptions...", flush=True)
smet_embs   = encode_texts(df_smet_eval["Description"].tolist(),
                            batch_size=CFG.batch_size, desc="SMET")
smet_scores = get_score_matrix(smet_embs)        # (N, 214)

smet_labels = mlb_full.transform(
    df_smet_eval["tcodes"].tolist()).astype(np.float32)

print(f"smet_scores shape   : {smet_scores.shape}")
print(f"smet_labels shape   : {smet_labels.shape}")
print(f"label density       : {smet_labels.mean():.4f}")

# ── 8. SMET metrics ───────────────────────────────────────────────
r_smet = full_metrics(smet_scores, smet_labels,
                      "SMET zero-shot (n={})  [paper: CE=13.96 RL=0.05 "
                      "LRAP=53.77% R@5=67.71%]".format(len(df_smet_eval)))

# ══════════════════════════════════════════════════════════════════
# C. SUMMARY
# ══════════════════════════════════════════════════════════════════
print("\n" + "═"*65)
print("ZERO-SHOT SUMMARY")
print("═"*65)
print(f"{'Eval Set':<30} {'LRAP':>7} {'RL':>7} {'CE':>7} "
      f"{'R@1':>6} {'R@5':>6} {'R@10':>6}")
print("─"*65)
for tag, r in [("KEV (n={})".format(len(df_kev)),   r_kev),
               ("SMET (n={})".format(len(df_smet_eval)), r_smet),
               ("SMET paper baseline",
                dict(lrap=0.5377, ranking_loss=0.05,
                     coverage_error=13.96, r1=None, r5=0.6771, r10=None))]:
    lrap = f"{r['lrap']:.4f}"
    rl   = f"{r['ranking_loss']:.4f}"
    ce   = f"{r['coverage_error']:.4f}"
    r1   = f"{r['r1']:.4f}" if r.get('r1') is not None else "  —  "
    r5   = f"{r['r5']:.4f}" if r.get('r5') is not None else "  —  "
    r10  = f"{r.get('r10',None):.4f}" \
           if r.get('r10') is not None else "  —  "
    print(f"{tag:<30} {lrap:>7} {rl:>7} {ce:>7} "
          f"{r1:>6} {r5:>6} {r10:>6}")
print("═"*65)

print("\n✓ Zero-shot evaluation complete.")

KEV eval rows       : 419
Unique T-codes      : 108
Rows with desc      : 419

Encoding KEV descriptions...
  KEV: 419/419
kev_scores shape    : (419, 214)
kev_labels shape    : (419, 214)
label density       : 0.0130

── KEV zero-shot (n=419) ──────────────────────────────────────────
  LRAP           : 0.1063
  Ranking Loss   : 0.2815
  Coverage Error : 101.5274
  R@1            : 0.0179
  R@3            : 0.0607
  R@5            : 0.0957
  R@10           : 0.1726

id2mitre loaded: 594 entries
name→T-code entries : 670
SMET rows mapped    : 302 / 303
Unique T-codes      : 40  ['T1005', 'T1007', 'T1016', 'T1040', 'T1055', 'T1059', 'T1068', 'T1078', 'T1083', 'T1110', 'T1136', 'T1176', 'T1189', 'T1190', 'T1195', 'T1203', 'T1204', 'T1211', 'T1213', 'T1485', 'T1491', 'T1498', 'T1499', 'T1505', 'T1518', 'T1528', 'T1529', 'T1531', 'T1539', 'T1543', 'T1547', 'T1548', 'T1552', 'T1557', 'T1562', 'T1565', 'T1566', 'T1574', 'T1598', 'T1606']

Encoding SMET descriptions...
  SMET: 302/302
smet_sc

In [5]:
import json
import pandas as pd
import ast
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

# ==========================================
# 1. Parse MITRE ATT&CK (Build attack_dict)
# ==========================================
print("Loading ATT&CK STIX data...")
with open("OSRs\ATTACK\enterprise-attack-v16.1.json", "r", encoding="utf-8") as f:
    stix_bundle = json.load(f)

attack_dict = {}
name_to_tcode = {} # Useful fallback for SMET parsing
parent_map = {}

# First pass: Extract all attack patterns
for obj in stix_bundle.get("objects", []):
    if obj.get("type") == "attack-pattern" and not obj.get("revoked") and not obj.get("x_mitre_deprecated"):
        t_code = None
        for ref in obj.get("external_references", []):
            if ref.get("source_name") == "mitre-attack":
                t_code = ref.get("external_id")
                break
        
        if t_code:
            # We use the raw STIX description. 
            # (Optional: you could strip HTML/Markdown tags here if desired)
            attack_dict[t_code] = obj.get("description", "")
            name_to_tcode[obj.get("name")] = t_code

# Second pass: Build parent map for sub-techniques (optional for this specific test, but good for alignment)
for obj in stix_bundle.get("objects", []):
    if obj.get("type") == "relationship" and obj.get("relationship_type") == "subtechnique-of":
        sub_id = obj.get("source_ref")
        parent_id = obj.get("target_ref")
        
        # Resolve STIX IDs to T-Codes
        sub_tcode = next((ext["external_id"] for o in stix_bundle["objects"] if o.get("id") == sub_id for ext in o.get("external_references", []) if ext.get("source_name") == "mitre-attack"), None)
        parent_tcode = next((ext["external_id"] for o in stix_bundle["objects"] if o.get("id") == parent_id for ext in o.get("external_references", []) if ext.get("source_name") == "mitre-attack"), None)
        
        if sub_tcode and parent_tcode:
            parent_map[sub_tcode] = parent_tcode

print(f"Loaded {len(attack_dict)} total techniques/sub-techniques from ATT&CK v16.1")

# ==========================================
# 2. Parse SMET Benchmark (Build smet_df)
# ==========================================
print("Loading SMET benchmark data...")
smet_raw = pd.read_excel("CVE_annotated_dataset.xlsx")

with open("id2mitre.json", "r", encoding="utf-8") as f:
    id2mitre = json.load(f)

smet_records = []
for _, row in smet_raw.iterrows():
    cve_id = row["ID"]
    description = row["Description"]
    
    # SMET stores techniques as a string representation of a list of names
    try:
        tech_names = ast.literal_eval(row["ATT&CK Techniques"])
    except:
        tech_names = []
        
    t_codes = []
    for name in tech_names:
        # 1. Try mapping via SMET's provided JSON
        mapped_id = None
        for key, vals in id2mitre.items():
            if name in vals:
                mapped_id = key
                break
        
        # 2. Fallback to STIX exact name match
        if not mapped_id:
            mapped_id = name_to_tcode.get(name)
            
        if mapped_id:
            # For this test, you can either keep sub-techniques or roll them up to parents.
            # We will roll up to match your earlier parent-only label space rule.
            final_id = parent_map.get(mapped_id, mapped_id)
            t_codes.append(final_id)
            
    if t_codes:
        # Deduplicate T-codes after parent rollup
        smet_records.append({
            "CVE_ID": cve_id,
            "Description": description,
            "T_Codes": list(set(t_codes))
        })

smet_df = pd.DataFrame(smet_records)
print(f"Successfully parsed {len(smet_df)} SMET CVEs with valid ATT&CK mappings.")

# ==========================================
# 3. Setup Model and Device
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model_name = "basel/ATTACK-BERT"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.to(device)
model.eval()

# ==========================================
# 4. Define Mean-Pooling Function
# ==========================================
def get_embeddings(text_list, batch_size=16):
    all_embeddings = []
    
    with torch.no_grad():
        for i in tqdm(range(0, len(text_list), batch_size), desc="Embedding"):
            batch_texts = text_list[i:i+batch_size]
            
            encoded_input = tokenizer(
                batch_texts, 
                padding=True, 
                truncation=True, 
                max_length=512, 
                return_tensors='pt'
            ).to(device)
            
            model_output = model(**encoded_input)
            
            attention_mask = encoded_input['attention_mask']
            token_embeddings = model_output.last_hidden_state
            
            input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
            sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
            sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
            
            mean_pooled = sum_embeddings / sum_mask
            mean_pooled = F.normalize(mean_pooled, p=2, dim=1)
            
            all_embeddings.append(mean_pooled.cpu())
            
    return torch.cat(all_embeddings, dim=0)

# ==========================================
# 5. Execute Zero-Shot Evaluation
# ==========================================
# Filter attack_dict to only include PARENT techniques (no ".") to match your thesis constraints
parent_attack_dict = {k: v for k, v in attack_dict.items() if "." not in k}
technique_ids = list(parent_attack_dict.keys())
technique_texts = list(parent_attack_dict.values())

cve_texts = smet_df['Description'].tolist()
cve_ground_truths = smet_df['T_Codes'].tolist()

print(f"\nEmbedding {len(technique_texts)} Parent ATT&CK Techniques...")
tech_embeddings = get_embeddings(technique_texts, batch_size=16)

print(f"\nEmbedding {len(cve_texts)} SMET CVEs...")
cve_embeddings = get_embeddings(cve_texts, batch_size=16)

# Compute similarity matrix
similarity_matrix = torch.matmul(cve_embeddings, tech_embeddings.T)

hits_at_1 = 0
hits_at_5 = 0
hits_at_10 = 0

for i in range(len(cve_texts)):
    true_labels = set(cve_ground_truths[i])
    
    # Get top 10 indices
    top_10_indices = torch.topk(similarity_matrix[i], k=10).indices.tolist()
    top_10_predictions = [technique_ids[idx] for idx in top_10_indices]
    
    # R@1
    if top_10_predictions[0] in true_labels:
        hits_at_1 += 1
        
    # R@5
    if len(true_labels.intersection(set(top_10_predictions[:5]))) > 0:
        hits_at_5 += 1
        
    # R@10
    if len(true_labels.intersection(set(top_10_predictions))) > 0:
        hits_at_10 += 1

n_cves = len(cve_texts)
print("\n" + "="*50)
print("ZERO-SHOT BI-ENCODER RESULTS (Untrained)")
print("="*50)
print(f"Total SMET CVEs Evaluated: {n_cves}")
print(f"Candidate Techniques (Parents Only): {len(technique_ids)}")
print("-" * 50)
print(f"Recall@1:  {(hits_at_1 / n_cves) * 100:.2f}%")
print(f"Recall@5:  {(hits_at_5 / n_cves) * 100:.2f}%")
print(f"Recall@10: {(hits_at_10 / n_cves) * 100:.2f}%")
print("="*50)

Loading ATT&CK STIX data...
Loaded 656 total techniques/sub-techniques from ATT&CK v16.1
Loading SMET benchmark data...
Successfully parsed 302 SMET CVEs with valid ATT&CK mappings.
Using device: cuda


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 28426.76it/s]



Embedding 203 Parent ATT&CK Techniques...


Embedding: 100%|██████████| 13/13 [00:01<00:00,  7.86it/s]



Embedding 302 SMET CVEs...


Embedding: 100%|██████████| 19/19 [00:00<00:00, 22.75it/s]


ZERO-SHOT BI-ENCODER RESULTS (Untrained)
Total SMET CVEs Evaluated: 302
Candidate Techniques (Parents Only): 203
--------------------------------------------------
Recall@1:  25.17%
Recall@5:  55.30%
Recall@10: 68.54%


In [11]:
from rank_bm25 import BM25Okapi
import numpy as np
import torch

# ==========================================
# 1. Build the BM25 Lexical Index
# (Run this right after you build your technique_texts list)
# ==========================================
print("\nBuilding BM25 Lexical Index...")
# Simple whitespace tokenizer for BM25
tokenized_corpus = [text.lower().split() for text in technique_texts]
bm25_model = BM25Okapi(tokenized_corpus)

# ==========================================
# 2. The Smart Hybrid RRF Function
# ==========================================
def hybrid_rrf_search(cve_text, semantic_scores_tensor, bm25_model, technique_ids, top_k=10, rrf_k=60):
    """
    Combines Semantic (ATT&CK-BERT) and Lexical (BM25) search using Reciprocal Rank Fusion.
    """
    # 1. Get Semantic Ranks (from the pre-computed similarity matrix)
    # Convert tensor to numpy for easier ranking
    semantic_scores = semantic_scores_tensor.cpu().numpy()
    
    # argsort gives ascending order, so we reverse it [::-1] for descending (highest score first)
    semantic_ranked_indices = np.argsort(semantic_scores)[::-1]
    
    # Create a dictionary mapping the index to its Semantic Rank (1st place = rank 1)
    semantic_ranks = {idx: rank + 1 for rank, idx in enumerate(semantic_ranked_indices)}
    
    # 2. Get Lexical Ranks (BM25)
    tokenized_query = cve_text.lower().split()
    bm25_scores = bm25_model.get_scores(tokenized_query)
    
    lexical_ranked_indices = np.argsort(bm25_scores)[::-1]
    lexical_ranks = {idx: rank + 1 for rank, idx in enumerate(lexical_ranked_indices)}
    
    # 3. Compute RRF Scores
    rrf_scores = {}
    for idx in range(len(technique_ids)):
        # Apply the RRF formula
        s_rank = semantic_ranks[idx]
        l_rank = lexical_ranks[idx]
        
        # If BM25 score is exactly 0.0 (no keyword overlap at all), penalize its rank to infinity
        if bm25_scores[idx] == 0.0:
            l_rank = float('inf')
            
        rrf_score = (1.0 / (rrf_k + s_rank)) + (1.0 / (rrf_k + l_rank))
        rrf_scores[idx] = rrf_score
        
    # 4. Sort by final RRF Score
    sorted_rrf_indices = sorted(rrf_scores.keys(), key=lambda x: rrf_scores[x], reverse=True)
    
    # Return the top K T-codes
    top_predictions = [technique_ids[idx] for idx in sorted_rrf_indices[:top_k]]
    return top_predictions

# ==========================================
# 3. Execute Hybrid Zero-Shot Evaluation
# (Replace your existing eval loop with this)
# ==========================================
print("\n" + "="*50)
print("Evaluating Smart Hybrid Search (ATT&CK-BERT + BM25 + RRF)")
print("="*50)

# Compute full semantic similarity matrix once (just like you did before)
similarity_matrix = torch.matmul(cve_embeddings, tech_embeddings.T)

hits_at_1 = 0
hits_at_5 = 0
hits_at_10 = 0

for i in tqdm(range(len(cve_texts)), desc="Hybrid Inference"):
    true_labels = set(cve_ground_truths[i])
    current_cve_text = cve_texts[i]
    
    # Get predictions using the smart RRF function
    top_10_predictions = hybrid_rrf_search(
        cve_text=current_cve_text,
        semantic_scores_tensor=similarity_matrix[i],
        bm25_model=bm25_model,
        technique_ids=technique_ids,
        top_k=10,
        rrf_k=60 # 60 is the industry standard constant for RRF
    )
    
    # R@1
    if top_10_predictions[0] in true_labels:
        hits_at_1 += 1
        
    # R@5
    if len(true_labels.intersection(set(top_10_predictions[:5]))) > 0:
        hits_at_5 += 1
        
    # R@10
    if len(true_labels.intersection(set(top_10_predictions))) > 0:
        hits_at_10 += 1

n_cves = len(cve_texts)
print("\n" + "="*50)
print("SMART HYBRID RESULTS (Zero-Shot)")
print("="*50)
print(f"Total SMET CVEs Evaluated: {n_cves}")
print("-" * 50)
print(f"Hybrid Hit Rate@1:  {(hits_at_1 / n_cves) * 100:.2f}%")
print(f"Hybrid Hit Rate@5:  {(hits_at_5 / n_cves) * 100:.2f}%")
print(f"Hybrid Hit Rate@10: {(hits_at_10 / n_cves) * 100:.2f}%")
print("="*50)


Building BM25 Lexical Index...

Evaluating Smart Hybrid Search (ATT&CK-BERT + BM25 + RRF)


Hybrid Inference:   0%|          | 0/302 [00:00<?, ?it/s]

Hybrid Inference: 100%|██████████| 302/302 [00:00<00:00, 603.12it/s]


SMART HYBRID RESULTS (Zero-Shot)
Total SMET CVEs Evaluated: 302
--------------------------------------------------
Hybrid Hit Rate@1:  32.45%
Hybrid Hit Rate@5:  63.91%
Hybrid Hit Rate@10: 76.16%


In [12]:
# ==========================================
# 6. KEV Gold Evaluation (same model, same index)
# ==========================================
print("Loading KEV gold labels...")
with open("OSRs/kev-07.28.2025_attack-16.1-enterprise.json", "r", encoding="utf-8") as f:
    kev_data = json.load(f)

# Group by CVE → (description, set of parent T-codes)
kev_by_cve = defaultdict(lambda: {"desc": "", "techs": set()})
for entry in kev_data["mapping_objects"]:
    cve_id  = entry.get("capability_id", "").strip()
    tc_raw  = entry.get("attack_object_id", "").strip()
    desc    = entry.get("capability_description", "").strip()
    if not cve_id.startswith("CVE-") or not tc_raw:
        continue
    # Roll sub-techniques up to parent (same logic as SMET)
    parent_tc = parent_map.get(tc_raw, tc_raw)
    # Only keep parent techniques (no dot) that exist in our index
    if "." not in parent_tc and parent_tc in parent_attack_dict:
        kev_by_cve[cve_id]["techs"].add(parent_tc)
    if desc:
        kev_by_cve[cve_id]["desc"] = desc

# Build eval lists — drop CVEs with empty desc or no mapped techniques
kev_cve_ids, kev_texts, kev_ground_truths = [], [], []
for cve_id, val in kev_by_cve.items():
    if val["desc"] and val["techs"]:
        kev_cve_ids.append(cve_id)
        kev_texts.append(val["desc"])
        kev_ground_truths.append(val["techs"])

print(f"KEV CVEs with desc + mapped techniques : {len(kev_texts)}")
print(f"Techniques outside parent index (dropped): "
      f"{sum(1 for e in kev_data['mapping_objects'] if '.' in e.get('attack_object_id',''))}"
      f" sub-technique rows")

# ── Embed KEV descriptions (technique index already built above) ──
print(f"\nEmbedding {len(kev_texts)} KEV CVE descriptions...")
kev_embeddings = get_embeddings(kev_texts, batch_size=16)

# Cosine similarity against same technique index
kev_sim = torch.matmul(kev_embeddings, tech_embeddings.T)

# ── Metrics (R@1, R@5, R@10 — same calculation as SMET) ──────────
kev_hits_1 = kev_hits_5 = kev_hits_10 = 0

for i in range(len(kev_texts)):
    true_labels  = kev_ground_truths[i]
    top_10_idx   = torch.topk(kev_sim[i], k=10).indices.tolist()
    top_10_preds = [technique_ids[idx] for idx in top_10_idx]

    if top_10_preds[0] in true_labels:
        kev_hits_1 += 1
    if true_labels.intersection(set(top_10_preds[:5])):
        kev_hits_5 += 1
    if true_labels.intersection(set(top_10_preds)):
        kev_hits_10 += 1

n_kev = len(kev_texts)
print("\n" + "="*50)
print("ZERO-SHOT BI-ENCODER RESULTS — KEV GOLD")
print("="*50)
print(f"Total KEV CVEs Evaluated    : {n_kev}")
print(f"Candidate Techniques        : {len(technique_ids)} parents")
print("-"*50)
print(f"Recall@1  : {kev_hits_1/n_kev*100:.2f}%")
print(f"Recall@5  : {kev_hits_5/n_kev*100:.2f}%")
print(f"Recall@10 : {kev_hits_10/n_kev*100:.2f}%")
print("="*50)

# ── Side-by-side comparison ───────────────────────────────────────
print("\n" + "="*50)
print("COMPARISON: SMET vs KEV (AttackBERT zero-shot)")
print("="*50)
print(f"{'Metric':<12} {'SMET':>10} {'KEV':>10}")
print("-"*35)
print(f"{'R@1':<12} {hits_at_1/n_cves*100:>9.2f}% {kev_hits_1/n_kev*100:>9.2f}%")
print(f"{'R@5':<12} {hits_at_5/n_cves*100:>9.2f}% {kev_hits_5/n_kev*100:>9.2f}%")
print(f"{'R@10':<12} {hits_at_10/n_cves*100:>9.2f}% {kev_hits_10/n_kev*100:>9.2f}%")
print(f"{'N':<12} {n_cves:>10} {n_kev:>10}")
print("="*50)

Loading KEV gold labels...
KEV CVEs with desc + mapped techniques : 419
Techniques outside parent index (dropped): 222 sub-technique rows

Embedding 419 KEV CVE descriptions...


Embedding: 100%|██████████| 27/27 [00:00<00:00, 55.47it/s]


ZERO-SHOT BI-ENCODER RESULTS — KEV GOLD
Total KEV CVEs Evaluated    : 419
Candidate Techniques        : 203 parents
--------------------------------------------------
Recall@1  : 8.11%
Recall@5  : 26.73%
Recall@10 : 41.05%

COMPARISON: SMET vs KEV (AttackBERT zero-shot)
Metric             SMET        KEV
-----------------------------------
R@1              32.45%      8.11%
R@5              63.91%     26.73%
R@10             76.16%     41.05%
N                   302        419


In [13]:
# ==========================================
# 4. Hybrid Evaluation on KEV Gold
# ==========================================
print("\nEvaluating Hybrid Search on KEV...")
kev_sim = torch.matmul(kev_embeddings, tech_embeddings.T)

kev_h1 = kev_h5 = kev_h10 = 0
for i in tqdm(range(len(kev_texts)), desc="KEV Hybrid"):
    true_labels = kev_ground_truths[i]
    top_10 = hybrid_rrf_search(
        cve_text=kev_texts[i],
        semantic_scores_tensor=kev_sim[i],
        bm25_model=bm25_model,
        technique_ids=technique_ids,
        top_k=10,
        rrf_k=60,
    )
    if top_10[0] in true_labels:                         kev_h1  += 1
    if true_labels.intersection(set(top_10[:5])):        kev_h5  += 1
    if true_labels.intersection(set(top_10)):            kev_h10 += 1

n_kev = len(kev_texts)
print("\n" + "="*55)
print("FINAL COMPARISON — Semantic vs Hybrid (Zero-Shot)")
print("="*55)
print(f"{'Metric':<12} {'SMET Sem':>10} {'SMET Hyb':>10} "
      f"{'KEV Sem':>10} {'KEV Hyb':>10}")
print("-"*55)
for metric, s_sem, s_hyb, k_sem, k_hyb in [
    ("R@1",  hits_at_1,  None,    kev_hits_1,  kev_h1),
    ("R@5",  hits_at_5,  None,    kev_hits_5,  kev_h5),
    ("R@10", hits_at_10, None,    kev_hits_10, kev_h10),
]:
    # pure semantic from your earlier run
    s_sem_pct = s_sem / n_cves * 100
    k_sem_pct = k_sem / n_kev * 100
    k_hyb_pct = k_hyb / n_kev * 100
    print(f"{metric:<12} {s_sem_pct:>9.2f}% {'(above)':>10} "
          f"{k_sem_pct:>9.2f}% {k_hyb_pct:>9.2f}%")
print(f"{'N':<12} {n_cves:>10} {'':>10} {n_kev:>10}")
print("="*55)


Evaluating Hybrid Search on KEV...


KEV Hybrid: 100%|██████████| 419/419 [00:00<00:00, 1814.16it/s]


FINAL COMPARISON — Semantic vs Hybrid (Zero-Shot)
Metric         SMET Sem   SMET Hyb    KEV Sem    KEV Hyb
-------------------------------------------------------
R@1              32.45%    (above)      8.11%     12.89%
R@5              63.91%    (above)     26.73%     26.25%
R@10             76.16%    (above)     41.05%     40.57%
N                   302                   419


In [15]:
# Cell A — Enrich Technique Texts with Mitigations + Examples

# ── Build mitigation text per technique from STIX ─────────────────
# Mitigations linked via "mitigates" relationships
tech_mitigations = defaultdict(list)   # tcode → [mitigation descriptions]
tech_detections  = defaultdict(list)   # tcode → [detection text]

# Index mitigations by STIX id
mitigations_by_id = {}
for obj in stix_bundle["objects"]:
    if obj.get("type") == "course-of-action":
        mitigations_by_id[obj["id"]] = obj.get("description", "")

# Follow "mitigates" relationships
for obj in stix_bundle["objects"]:
    if obj.get("type") != "relationship":
        continue
    if obj.get("relationship_type") != "mitigates":
        continue
    tgt_id   = obj.get("target_ref", "")
    src_id   = obj.get("source_ref", "")
    tgt_obj  = next((o for o in stix_bundle["objects"]
                     if o.get("id") == tgt_id), None)
    if not tgt_obj:
        continue
    tc = next((r["external_id"] for r in
               tgt_obj.get("external_references", [])
               if r.get("source_name") == "mitre-attack"), None)
    if tc and src_id in mitigations_by_id:
        desc = mitigations_by_id[src_id]
        if desc:
            tech_mitigations[tc].append(desc)

# Extract x_mitre_detection field (detection guidance)
for obj in stix_bundle["objects"]:
    if obj.get("type") != "attack-pattern":
        continue
    tc = next((r["external_id"] for r in
               obj.get("external_references", [])
               if r.get("source_name") == "mitre-attack"), None)
    det = obj.get("x_mitre_detection", "")
    if tc and det:
        tech_detections[tc].append(det)

print(f"Techniques with mitigation text : "
      f"{sum(1 for v in tech_mitigations.values() if v)}")
print(f"Techniques with detection text  : "
      f"{sum(1 for v in tech_detections.values() if v)}")

# ── Rebuild technique texts ────────────────────────────────────────
def build_enriched_technique_text(tc, max_mit_chars=300, max_det_chars=300):
    name  = name_to_tcode.get(tc, tc)   # use tcode as fallback
    # get name properly
    tname = next((obj.get("name","") for obj in stix_bundle["objects"]
                  if obj.get("type") == "attack-pattern"
                  and any(r.get("external_id") == tc
                          for r in obj.get("external_references", [])
                          if r.get("source_name") == "mitre-attack")), tc)
    desc  = attack_dict.get(tc, "")
    # clean citations
    desc  = re.sub(r'\(Citation:[^)]+\)', '', desc).strip()
    # first 3 sentences
    sents = re.split(r'(?<=[.!?])\s+', desc)
    desc_short = " ".join(sents[:3]).strip()

    # sub-technique names
    subs = [obj.get("name","") for obj in stix_bundle["objects"]
            if obj.get("type") == "attack-pattern"
            and any(r.get("external_id","").startswith(tc + ".")
                    for r in obj.get("external_references", [])
                    if r.get("source_name") == "mitre-attack")]

    # mitigation summary (truncated)
    mit_text = " ".join(tech_mitigations.get(tc, []))
    mit_text = re.sub(r'\(Citation:[^)]+\)', '', mit_text)[:max_mit_chars].strip()

    # detection summary (truncated)
    det_text = " ".join(tech_detections.get(tc, []))
    det_text = re.sub(r'\(Citation:[^)]+\)', '', det_text)[:max_det_chars].strip()

    parts = [f"[{tc}] {tname}."]
    if desc_short:
        parts.append(desc_short)
    if subs:
        parts.append(f"Sub-techniques: {', '.join(subs[:8])}.")
    if mit_text:
        parts.append(f"Mitigations: {mit_text}")
    if det_text:
        parts.append(f"Detection: {det_text}")
    return " ".join(parts)

import re
from collections import defaultdict

# Rebuild index texts
technique_texts_enriched = [
    build_enriched_technique_text(tc) for tc in technique_ids]

# Compare lengths
orig_len  = np.mean([len(t) for t in technique_texts])
enrich_len = np.mean([len(t) for t in technique_texts_enriched])
print(f"\nAvg technique text length:")
print(f"  Before enrichment : {orig_len:.0f} chars")
print(f"  After enrichment  : {enrich_len:.0f} chars")
print(f"\nSample (T1190):")
idx_t1190 = technique_ids.index("T1190")
print(technique_texts_enriched[idx_t1190][:400])

# ── Re-embed technique index ───────────────────────────────────────
print("\nRe-embedding enriched technique index...")
tech_embeddings_enriched = get_embeddings(
    technique_texts_enriched, batch_size=16)
tech_embeddings_enriched_np = tech_embeddings_enriched.numpy()

# ── Rebuild BM25 on enriched texts ────────────────────────────────
print("Rebuilding BM25 on enriched texts...")
tokenized_corpus_enriched = [t.lower().split()
                              for t in technique_texts_enriched]
bm25_enriched = BM25Okapi(tokenized_corpus_enriched)
print("✓ Enriched index ready.")

Techniques with mitigation text : 729
Techniques with detection text  : 725

Avg technique text length:
  Before enrichment : 1301 chars
  After enrichment  : 1098 chars

Sample (T1190):
[T1190] Exploit Public-Facing Application. Adversaries may attempt to exploit a weakness in an Internet-facing host or system to initially access a network. The weakness in the system can be a software bug, a temporary glitch, or a misconfiguration. Exploited applications are often websites/web servers, but can also include databases (like SQL), standard services (like SMB or SSH), network device 

Re-embedding enriched technique index...


Embedding: 100%|██████████| 13/13 [00:01<00:00, 10.45it/s]

Rebuilding BM25 on enriched texts...
✓ Enriched index ready.


In [16]:
# Cell B — Enrich KEV Descriptions from NVD 2.0 + Re-evaluate

import glob, json

# ── 1. Build CVE-ID → full NVD description lookup ─────────────────
print("Building NVD lookup (this takes ~30s)...")
nvd_desc_lookup = {}   # "CVE-2021-1234" → full English description

nvd_files = sorted(glob.glob("OSRs/NVD/nvdcve-2.0-*.json"))
print(f"  NVD files found: {len(nvd_files)}")

for fpath in nvd_files:
    with open(fpath, encoding="utf-8") as f:
        data = json.load(f)
    for entry in data.get("vulnerabilities", []):
        cve_obj = entry.get("cve", {})
        cve_id  = cve_obj.get("id", "")
        for d in cve_obj.get("descriptions", []):
            if d.get("lang") == "en":
                txt = d.get("value", "").strip()
                if txt and not txt.startswith("** REJECT"):
                    nvd_desc_lookup[cve_id] = txt
                break

print(f"  NVD descriptions loaded: {len(nvd_desc_lookup):,}")

# ── 2. Enrich KEV descriptions ────────────────────────────────────
kev_texts_enriched = []
enriched_count = 0
for cve_id, text in zip(kev_cve_ids, kev_texts):
    nvd_text = nvd_desc_lookup.get(cve_id, "")
    if nvd_text and len(nvd_text) > len(text):
        # Prepend short KEV desc + full NVD desc
        kev_texts_enriched.append(f"{text} {nvd_text}")
        enriched_count += 1
    else:
        kev_texts_enriched.append(text)

print(f"\nKEV descriptions enriched via NVD : {enriched_count} / {len(kev_texts)}")
avg_before = np.mean([len(t) for t in kev_texts])
avg_after  = np.mean([len(t) for t in kev_texts_enriched])
print(f"Avg desc length before : {avg_before:.0f} chars")
print(f"Avg desc length after  : {avg_after:.0f} chars")

# ── 3. Re-embed enriched KEV descriptions ────────────────────────
print("\nRe-embedding enriched KEV descriptions...")
kev_embs_enriched = get_embeddings(kev_texts_enriched, batch_size=16)

# ── 4. Evaluate: 4 combinations ──────────────────────────────────
configs = [
    ("Semantic  | orig  tech + orig  KEV",
     kev_embeddings,     tech_embeddings,          technique_texts,
     bm25_model,         kev_texts),
    ("Semantic  | enr   tech + enr   KEV",
     kev_embs_enriched,  tech_embeddings_enriched, technique_texts_enriched,
     bm25_enriched,      kev_texts_enriched),
    ("Hybrid RRF| orig  tech + orig  KEV",
     kev_embeddings,     tech_embeddings,          technique_texts,
     bm25_model,         kev_texts),
    ("Hybrid RRF| enr   tech + enr   KEV",
     kev_embs_enriched,  tech_embeddings_enriched, technique_texts_enriched,
     bm25_enriched,      kev_texts_enriched),
]

use_hybrid = [False, False, True, True]

print("\n" + "="*65)
print("KEV ABLATION — Enrichment Impact")
print("="*65)
print(f"{'Config':<42} {'R@1':>7} {'R@5':>7} {'R@10':>7}")
print("─"*65)

for (label, q_emb, t_emb, t_texts, bm25_idx, q_texts), hybrid in \
        zip(configs, use_hybrid):
    sim = torch.matmul(q_emb, t_emb.T)
    h1 = h5 = h10 = 0
    for i in range(len(kev_texts)):
        true_labels = kev_ground_truths[i]
        if hybrid:
            top10 = hybrid_rrf_search(
                q_texts[i], sim[i], bm25_idx, technique_ids, top_k=10)
        else:
            top10 = [technique_ids[j]
                     for j in torch.topk(sim[i], 10).indices.tolist()]
        if top10[0] in true_labels:               h1  += 1
        if true_labels.intersection(top10[:5]):   h5  += 1
        if true_labels.intersection(top10):       h10 += 1
    n = len(kev_texts)
    print(f"{label:<42} {h1/n*100:>6.2f}% {h5/n*100:>6.2f}% {h10/n*100:>6.2f}%")

# Also re-run SMET with enriched technique index for comparison
print("─"*65)
sim_smet_enr = torch.matmul(cve_embeddings, tech_embeddings_enriched.T)
s1 = s5 = s10 = 0
for i in range(len(cve_texts)):
    true_labels = set(cve_ground_truths[i])
    top10 = hybrid_rrf_search(
        cve_texts[i], sim_smet_enr[i], bm25_enriched,
        technique_ids, top_k=10)
    if top10[0] in true_labels:                    s1  += 1
    if true_labels.intersection(set(top10[:5])):   s5  += 1
    if true_labels.intersection(set(top10)):       s10 += 1
n_s = len(cve_texts)
print(f"{'SMET Hybrid | enr tech (baseline: 63.91%)':<42} "
      f"{s1/n_s*100:>6.2f}% {s5/n_s*100:>6.2f}% {s10/n_s*100:>6.2f}%")
print("="*65)

Building NVD lookup (this takes ~30s)...
  NVD files found: 25
  NVD descriptions loaded: 333,022

KEV descriptions enriched via NVD : 375 / 419
Avg desc length before : 62 chars
Avg desc length after  : 395 chars

Re-embedding enriched KEV descriptions...


Embedding: 100%|██████████| 27/27 [00:01<00:00, 15.52it/s]



KEV ABLATION — Enrichment Impact
Config                                         R@1     R@5    R@10
─────────────────────────────────────────────────────────────────
Semantic  | orig  tech + orig  KEV           8.11%  26.73%  41.05%
Semantic  | enr   tech + enr   KEV           8.11%  29.83%  45.35%
Hybrid RRF| orig  tech + orig  KEV          12.89%  26.25%  40.57%
Hybrid RRF| enr   tech + enr   KEV          12.89%  28.64%  47.73%
─────────────────────────────────────────────────────────────────
SMET Hybrid | enr tech (baseline: 63.91%)   32.78%  62.58%  73.84%


In [31]:
from sentence_transformers import SentenceTransformer, util
import json
import torch

# ==========================================
# 1. Setup Model (The Correct Way)
# ==========================================
print("Loading AttackBERT via SentenceTransformers...")
# This wrapper automatically handles tokenization and mean-pooling, giving us the .encode() method
model = SentenceTransformer("basel/Attack-BERT")

# Ensure we have our techniques to compare against (assuming parent_attack_dict is still in memory from earlier)
# If you restarted your kernel, you'll need to re-run the STIX parsing cell first!
technique_ids = list(parent_attack_dict.keys())
technique_texts = list(parent_attack_dict.values())

print("Embedding ATT&CK technique dictionary...")
# Compute the database of vectors once
tech_embeddings = model.encode(technique_texts, convert_to_tensor=True)

# ==========================================
# 2. Load the Symbolic Mappings (ATT&CK -> NIST)
# ==========================================
def load_nist_mappings(filepath="OSRs/NIST800-53-layer-navigator/nist_800_53-rev5_attack-16.1-enterprise_json.json"):
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            return json.load(f)
    except FileNotFoundError:
        print(f"Warning: {filepath} not found. Using mock CTID mapping for demonstration.")
        return {
            "T1190": ["AC-3 (Access Enforcement)", "SI-4 (Information System Monitoring)", "SI-3 (Malicious Code Protection)"],
            "T1059": ["CM-6 (Configuration Settings)", "AU-2 (Event Logging)"],
            "T1566": ["AT-2 (Security Awareness Training)", "SC-15 (Collaborative Computing Devices)"],
            "T1078": ["AC-2 (Account Management)", "IA-2 (Identification and Authentication)"],
            "T1195": ["SA-12 (Supply Chain Protection)", "SI-4 (Information System Monitoring)"]
        }

nist_map = load_nist_mappings()

# ==========================================
# 3. Define the End-to-End Prediction Function
# ==========================================
def predict_threat_model(cve_text, top_k_techniques=3):
    """
    Takes a raw CVE string, predicts ATT&CK techniques via Bi-Encoder,
    and maps them to NIST controls via Symbolic lookup.
    """
    # 1. Neural Retrieval (CVE -> ATT&CK)
    cve_embedding = model.encode(cve_text, convert_to_tensor=True)
    
    # Compute Cosine Similarity using SentenceTransformers utility
    cos_scores = util.cos_sim(cve_embedding, tech_embeddings)[0]
    
    # Get Top-K
    top_results = torch.topk(cos_scores, k=top_k_techniques)
    
    predicted_techniques = []
    for score, idx in zip(top_results.values, top_results.indices):
        t_code = technique_ids[idx.item()]
        predicted_techniques.append({
            "t_code": t_code,
            "score": score.item()
        })
        
    # 2. Symbolic Lookup (ATT&CK -> NIST)
    final_report = {
        "cve_text": cve_text,
        "predicted_attack_path": [],
        "recommended_nist_controls": set()
    }
    
    for tech in predicted_techniques:
        t_code = tech["t_code"]
        controls = nist_map.get(t_code, ["No direct NIST mapping found in CTID data."])
        
        final_report["predicted_attack_path"].append({
            "Technique": t_code,
            "Confidence": f"{tech['score']:.2f}"
        })
        
        for control in controls:
            final_report["recommended_nist_controls"].add(control)
            
    final_report["recommended_nist_controls"] = list(final_report["recommended_nist_controls"])
    return final_report

# ==========================================
# 4. Execute the Pipeline
# ==========================================
# A classic Supply Chain / Phishing hybrid CVE text
sample_cve = "An issue was discovered in the web interface of the router. An unauthenticated attacker can send a crafted HTTP request containing a malicious payload via a phishing email to execute arbitrary operating system commands as the root user."

print("\n" + "="*50)
print("AUTOMATED THREAT MODELING PIPELINE: EXECUTION")
print("="*50)

report = predict_threat_model(sample_cve, top_k_techniques=2)

print(f"\n[INPUT CVE]:\n{report['cve_text']}\n")

print("[NEURAL PREDICTION: ATT&CK TECHNIQUES]:")
for tech in report['predicted_attack_path']:
    print(f" -> {tech['Technique']} (Confidence Score: {tech['Confidence']})")

print("\n[SYMBOLIC LOOKUP: NIST 800-53 CONTROLS]:")
for control in report['recommended_nist_controls']:
    print(f" -> {control}")
print("="*50)

Loading AttackBERT via SentenceTransformers...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 37132.60it/s]


Embedding ATT&CK technique dictionary...

AUTOMATED THREAT MODELING PIPELINE: EXECUTION

[INPUT CVE]:
An issue was discovered in the web interface of the router. An unauthenticated attacker can send a crafted HTTP request containing a malicious payload via a phishing email to execute arbitrary operating system commands as the root user.

[NEURAL PREDICTION: ATT&CK TECHNIQUES]:
 -> T1557 (Confidence Score: 0.51)
 -> T1599 (Confidence Score: 0.39)

[SYMBOLIC LOOKUP: NIST 800-53 CONTROLS]:
 -> No direct NIST mapping found in CTID data.


In [ ]:
!pip install datasets

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
     ------------------------------------- 527.0/527.0 kB 11.0 MB/s eta 0:00:00
     --------------------------------------- 27.5/27.5 MB 11.7 MB/s eta 0:00:00
     ---------------------------------------- 120.0/120.0 kB ? eta 0:00:00
     -------------------------------------- 144.5/144.5 kB 8.9 MB/s eta 0:00:00
     ------------------------------------- 462.9/462.9 kB 30.2 MB/s eta 0:00:00
     ---------------------------------------- 67.5/67.5 kB ? eta 0:00:00
     ---------------------------------------- 44.1/44.1 kB ? eta 0:00:00
     ---------------------------------------- 46.0/46.0 kB ? eta 0:00:00
     ---------------------------------------- 41.6/41.6 kB ? eta 0:00:00
     ---------------------------------------- 87.5/87.5 kB ? eta 0:00:00



[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
